In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "app":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABELS = ["negative", "neutral", "positive"]
TR_LABELS = {
    "negative": "Negatif",
    "neutral": "Tarafsız",
    "positive": "Pozitif",
}

def read_table(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    if path.suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(f"Desteklenmeyen dosya türü: {path}")

def dataset_summary(name: str, path: str, label_col: str) -> dict:
    df = read_table(PROJECT_ROOT / path)
    counts = df[label_col].astype(str).str.lower().value_counts()

    row = {
        "Veri seti": name,
        "N": int(len(df)),
        "Negatif": int(counts.get("negative", 0)),
        "Tarafsız": int(counts.get("neutral", 0)),
        "Pozitif": int(counts.get("positive", 0)),
    }

    print(f"\n{name}")
    print(f"Dosya: {path}")
    print(f"N: {row['N']}")
    print(f"Negatif: {row['Negatif']}")
    print(f"Tarafsız: {row['Tarafsız']}")
    print(f"Pozitif: {row['Pozitif']}")

    return row

rows = []

train_row = dataset_summary(
    "Eğitim",
    "db/splits/plain_sentiment_v1/train_df.parquet",
    "label",
)
val_row = dataset_summary(
    "Doğrulama",
    "db/splits/plain_sentiment_v1/val_df.parquet",
    "label",
)
test_row = dataset_summary(
    "Test",
    "db/splits/plain_sentiment_v1/test_df.parquet",
    "label",
)

main_pool_row = {
    "Veri seti": "Ana havuz",
    "N": train_row["N"] + val_row["N"] + test_row["N"],
    "Negatif": train_row["Negatif"] + val_row["Negatif"] + test_row["Negatif"],
    "Tarafsız": train_row["Tarafsız"] + val_row["Tarafsız"] + test_row["Tarafsız"],
    "Pozitif": train_row["Pozitif"] + val_row["Pozitif"] + test_row["Pozitif"],
}

rows.extend([main_pool_row, train_row, val_row, test_row])

rows.append(dataset_summary(
    "S&P 500 dış test",
    "outputs/finetuned_model_results/finbert_target_finetuned_seed42/external_evaluation/sp500_external/predictions.csv",
    "gold_label",
))

rows.append(dataset_summary(
    "Sentetik test",
    "outputs/finetuned_model_results/finbert_target_finetuned_seed42/external_evaluation/synthetic_external/predictions.csv",
    "gold_label",
))

rows.append(dataset_summary(
    "Reuters dış test",
    "outputs/finetuned_model_results/finbert_target_finetuned_seed42/external_evaluation/reuters_external/predictions.csv",
    "gold_label",
))

summary_df = pd.DataFrame(rows)

print("\nTablo B-2 Veri seti büyüklükleri ve sınıf dağılımları")
display(summary_df)


Eğitim
Dosya: db/splits/plain_sentiment_v1/train_df.parquet
N: 12950
Negatif: 1820
Tarafsız: 8137
Pozitif: 2993

Doğrulama
Dosya: db/splits/plain_sentiment_v1/val_df.parquet
N: 1432
Negatif: 215
Tarafsız: 929
Pozitif: 288

Test
Dosya: db/splits/plain_sentiment_v1/test_df.parquet
N: 2386
Negatif: 358
Tarafsız: 1549
Pozitif: 479

S&P 500 dış test
Dosya: outputs/finetuned_model_results/finbert_target_finetuned_seed42/external_evaluation/sp500_external/predictions.csv
N: 1060
Negatif: 323
Tarafsız: 339
Pozitif: 398

Sentetik test
Dosya: outputs/finetuned_model_results/finbert_target_finetuned_seed42/external_evaluation/synthetic_external/predictions.csv
N: 3000
Negatif: 1000
Tarafsız: 1000
Pozitif: 1000

Reuters dış test
Dosya: outputs/finetuned_model_results/finbert_target_finetuned_seed42/external_evaluation/reuters_external/predictions.csv
N: 5000
Negatif: 1579
Tarafsız: 1146
Pozitif: 2275

Tablo B-2 Veri seti büyüklükleri ve sınıf dağılımları


,Veri seti,N,Negatif,Tarafsız,Pozitif
0,Ana havuz,16768,2393,10615,3760
1,Eğitim,12950,1820,8137,2993
2,Doğrulama,1432,215,929,288
3,Test,2386,358,1549,479
4,S&P 500 dış test,1060,323,339,398
5,Sentetik test,3000,1000,1000,1000
6,Reuters dış test,5000,1579,1146,2275
